In [1]:
import pandas as pd                                   # pandas = the table tool; "pd" is its short name

matches = pd.read_csv("../exports/dim_matches.csv")   # read the CSV into a table called "matches"
matches.head()                                        # show the first 5 rows

,match_id,season,season_year,match_date,match_number,playoff_order,is_final,match_label,venue_name,city,...,match_result,match_winner,winner_runs,winner_wickets,method,target_overs,target_runs,player_of_match,champion,runner_up
0,1237177,2020/21,2020,2020-11-05,NaN,1.0,False,Playoff 1: Mumbai Indians vs Delhi Capitals,Dubai International Cricket Stadium,Dubai,...,win,Mumbai Indians,57.0,NaN,NaN,20.0,201.0,JJ Bumrah,Mumbai Indians,Delhi Capitals
1,1216507,2020/21,2020,2020-10-11,26.0,NaN,False,Match 26: Sunrisers Hyderabad vs Rajasthan Royals,Dubai International Cricket Stadium,Dubai,...,win,Rajasthan Royals,NaN,5.0,NaN,20.0,159.0,R Tewatia,Mumbai Indians,Delhi Capitals
2,1216508,2020/21,2020,2020-09-23,5.0,NaN,False,Match 5: Mumbai Indians vs Kolkata Knight Riders,Sheikh Zayed Stadium,Abu Dhabi,...,win,Mumbai Indians,49.0,NaN,NaN,20.0,196.0,RG Sharma,Mumbai Indians,Delhi Capitals
3,1216528,2020/21,2020,2020-10-13,29.0,NaN,False,Match 29: Chennai Super Kings vs Sunrisers Hyd...,Dubai International Cricket Stadium,Dubai,...,win,Chennai Super Kings,20.0,NaN,NaN,20.0,168.0,RA Jadeja,Mumbai Indians,Delhi Capitals
4,1216538,2020/21,2020,2020-10-04,17.0,NaN,False,Match 17: Mumbai Indians vs Sunrisers Hyderabad,Sharjah Cricket Stadium,Sharjah,...,win,Mumbai Indians,34.0,NaN,NaN,20.0,209.0,TA Boult,Mumbai Indians,Delhi Capitals


In [2]:
batting = pd.read_csv("../exports/fct_batting_innings.csv")              # read the batting innings table

batting = batting.merge(matches[["match_id", "season_year"]],             # JOIN to matches on match_id,
                        on="match_id")                                    # to add each innings' season_year

sixes = batting.groupby("season_year")["sixes"].sum()                    # GROUP BY season: total sixes
games = matches.groupby("season_year")["match_id"].count()               # GROUP BY season: number of matches

season = pd.DataFrame({"sixes": sixes, "matches": games})                # put both into one table, one row per season
season["sixes_per_match"] = (season["sixes"] / season["matches"]).round(2)   # the rate we will forecast

season                                                                     # show the table

,sixes,matches,sixes_per_match
season_year,,,
2008,622,58,10.72
2009,506,57,8.88
2010,585,60,9.75
2011,639,73,8.75
2012,731,74,9.88
2013,674,76,8.87
2014,714,60,11.90
2015,692,59,11.73
2016,638,60,10.63


In [3]:
train = season.loc[:2024]                                  # seasons 2008 to 2024: the models learn from these
test = season.loc[2025:]                                   # seasons 2025 and 2026: hidden, used to check accuracy

results = test[["sixes_per_match"]].copy()                 # start a results table with the real 2025 and 2026 values
results["naive"] = train["sixes_per_match"].iloc[-1]       # naive model: both seasons = the last known value (2024)

results                                                    # show actual vs naive forecast

,sixes_per_match,naive
season_year,,
2025,17.57,17.75
2026,19.27,17.75


In [12]:
import numpy as np                                    # numpy = maths tools

years = train.index                                   # the seasons 2008 to 2024
values = train["sixes_per_match"]                     # their sixes per match

slope, intercept = np.polyfit(years, values, 1)       # find the best straight line (the "1" means a straight line)

results["trend"] = (intercept + slope * results.index).round(2)   # line formula for 2025 and 2026

print("Line goes up each season by:", round(slope, 2))
results

Line goes up each season by: 0.41


,sixes_per_match,naive,trend
season_year,,,
2025,17.57,17.75,15.59
2026,19.27,17.75,16.00


In [13]:
pre = train.loc[:2022]                                      # seasons before the rule change: 2008 to 2022
slope_pre, intercept_pre = np.polyfit(pre.index, pre["sixes_per_match"], 1)   # trend line on those seasons only

post = train.loc[2023:]                                     # seasons after the rule (in training): 2023, 2024
line_at_post = intercept_pre + slope_pre * post.index       # where the old trend line says they "should" be
uplift = (post["sixes_per_match"] - line_at_post).mean()    # average gap above the line = the rule's effect

results["trend_plus_rule"] = (intercept_pre + slope_pre * results.index + uplift).round(2)   # old line + uplift

print("Pre-2023 trend per season:", round(slope_pre, 2))
print("Uplift from rule change:", round(uplift, 2))
results

Pre-2023 trend per season: 0.32
Uplift from rule change: 2.51


,sixes_per_match,naive,trend,trend_plus_rule
season_year,,,,
2025,17.57,17.75,15.59,16.95
2026,19.27,17.75,16.00,17.27


In [14]:
actual = results["sixes_per_match"]                                   # the real 2025 and 2026 values

naive_error = (results["naive"] - actual).abs().mean()                # naive: average size of the miss
trend_error = (results["trend"] - actual).abs().mean()                # straight line: average size of the miss
rule_error  = (results["trend_plus_rule"] - actual).abs().mean()      # trend + rule: average size of the miss

print("Naive MAE:          ", round(naive_error, 2))
print("Trend MAE:          ", round(trend_error, 2))
print("Trend + rule MAE:   ", round(rule_error, 2))

Naive MAE:           0.85
Trend MAE:           2.62
Trend + rule MAE:    1.31


In [15]:
first_value = train["sixes_per_match"].iloc[0]                 # 2008 value (first row of training)
last_value = train["sixes_per_match"].iloc[-1]                 # 2024 value (last row of training)
seasons_between = train.index[-1] - train.index[0]             # 2024 - 2008 = 16 seasons

drift = (last_value - first_value) / seasons_between           # average change per season

results["naive_drift"] = (last_value + drift * (results.index - 2024)).round(2)   # last value + growth per season ahead

drift_error = (results["naive_drift"] - actual).abs().mean()   # score it the same way

print("Average change per season:", round(drift, 2))
print("Naive + drift MAE:", round(drift_error, 2))
results

Average change per season: 0.44
Naive + drift MAE: 0.63


,sixes_per_match,naive,trend,trend_plus_rule,naive_drift
season_year,,,,,
2025,17.57,17.75,15.59,16.95,18.19
2026,19.27,17.75,16.00,17.27,18.63


In [18]:
all_values = season["sixes_per_match"]                                     # all 19 seasons, 2008 to 2026

drift_all = (all_values.iloc[-1] - all_values.iloc[0]) / (season.index[-1] - season.index[0])   # average growth per season, all data
spread = all_values.diff().std()                                           # how much seasons usually jump around year to year

forecast = pd.DataFrame(index=[2027, 2028])                                # empty table for the 2 future seasons
steps = forecast.index - 2026                                              # 1 season ahead, 2 seasons ahead

forecast["forecast"] = (all_values.iloc[-1] + drift_all * steps).round(2)                      # last value + growth
forecast["low_80"] = (forecast["forecast"] - 1.28 * spread * np.sqrt(steps)).round(2)          # bottom of the 80% range
forecast["high_80"] = (forecast["forecast"] + 1.28 * spread * np.sqrt(steps)).round(2)         # top of the 80% range

output = pd.concat([all_values.rename("actual"), forecast], axis=1)        # stack actuals and forecasts into one table
output.loc[2026, ["forecast", "low_80", "high_80"]] = output.loc[2026, "actual"]   # start the 3 forecast lines at the 2026 actual
output.to_csv("../exports/forecast_sixes.csv", index_label="season_year")  # save for Power BI

print("Average growth per season:", round(drift_all, 2))
output.tail(4)                                                             # show 2025, 2026 (actual) and 2027, 2028 (forecast)

Average growth per season: 0.47


,actual,forecast,low_80,high_80
2025,17.57,NaN,NaN,NaN
2026,19.27,19.27,19.27,19.27
2027,NaN,19.74,17.68,21.80
2028,NaN,20.22,17.30,23.14
